In [85]:
# CELL 1: Setup and load AUTO_v3 as v6 input

import pandas as pd
from pathlib import Path

BASE    = Path("/Users/davekokel/Projects/carp_v2")
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

OUT_SUFFIX = "_v6"

def out_csv(name: str) -> Path:
    return WORKING / f"{name}{OUT_SUFFIX}.csv"

auto_v3_path = WORKING / "imaging_roi_annotations_AUTO_v3.csv"
df = pd.read_csv(auto_v3_path)

df.shape, df.columns.tolist()

((976, 32),
 ['roi_dir',
  'date_experiment',
  'fish',
  'roi_name',
  'date_born',
  'parent_female',
  'parent_male',
  'genotype_base_codes',
  'genotype_allele_codes',
  'genotype_pretty',
  'genotype_marker_fluor_codes',
  'genotype_marker_tag_codes',
  'treatment_plasmid_base_codes',
  'treatment_rna_base_codes',
  'treatment_marker_fluor_codes',
  'treatment_marker_tag_codes',
  'all_marker_fluor_codes',
  'additional plasmids injected',
  'additional mRNAs injected',
  'additonal proteins injected',
  'additonal dye and chemicals',
  'Date born',
  'ZF female genotype',
  'ZF male genotype',
  'Data location',
  'plate_id_filled',
  'slot_id_filled',
  'roi_index_within_slot',
  'roi_code',
  'roi_dir_norm',
  'segments',
  'implied_date'])

In [86]:
# CELL 2: Strong blank detection + slug_infer from roi_dir

import math
import re

def is_blank(val: object) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    s = str(val).strip().lower()
    return s in ["", "nan", "none", "null", "na", "n/a", "-", "—", "."]

# define experiment slug from roi_dir: Foundation/ExperimentName
def extract_slug(path: str) -> str:
    p = str(path).replace("\\","/").rstrip("/")
    parts = p.split("/")
    # /.../Korra_Foundation/20250522_skittlez/fish1/roi1
    if len(parts) >= 4:
        foundation = parts[-4]
        experiment = parts[-3]
        return f"{foundation}/{experiment}"
    return p

df["roi_dir"]   = df["roi_dir"].astype(str).str.strip()
df["slug_infer"] = df["roi_dir"].apply(extract_slug)

df["slug_infer"].nunique(), df["slug_infer"].head()

(16,
 0    Korra_Foundation/20250428_mem_histone
 1    Korra_Foundation/20250428_mem_histone
 2    Korra_Foundation/20250429_mem_cytosol
 3    Korra_Foundation/20250429_mem_cytosol
 4    Korra_Foundation/20250429_mem_cytosol
 Name: slug_infer, dtype: object)

In [87]:
# CELL 3: Slug-level inference of parents and treatments (holes only)

slug_debug = []

for slug, sub in df.groupby("slug_infer"):
    n = len(sub)

    # parents
    pf_non = sub["parent_female"].dropna().astype(str).str.strip()
    pm_non = sub["parent_male"].dropna().astype(str).str.strip()

    can_infer_parents = (pf_non[~pf_non.apply(is_blank)].size > 0) and (pm_non[~pm_non.apply(is_blank)].size > 0)
    parent_female_cand = pf_non[~pf_non.apply(is_blank)].mode().iloc[0] if can_infer_parents else None
    parent_male_cand   = pm_non[~pm_non.apply(is_blank)].mode().iloc[0] if can_infer_parents else None

    # treatments
    tp_non = sub["treatment_plasmid_base_codes"].dropna().astype(str).str.strip()
    tr_non = sub["treatment_rna_base_codes"].dropna().astype(str).str.strip()

    can_infer_tp = tp_non[~tp_non.apply(is_blank)].size > 0
    can_infer_tr = tr_non[~tr_non.apply(is_blank)].size > 0

    tp_cand = tp_non[~tp_non.apply(is_blank)].mode().iloc[0] if can_infer_tp else None
    tr_cand = tr_non[~tr_non.apply(is_blank)].mode().iloc[0] if can_infer_tr else None

    # fill holes for this slug
    # parents: only when both are blank
    if can_infer_parents:
        mask_parent_holes = (
            (df["slug_infer"] == slug) &
            df["parent_female"].apply(is_blank) &
            df["parent_male"].apply(is_blank)
        )
        df.loc[mask_parent_holes, "parent_female"] = parent_female_cand
        df.loc[mask_parent_holes, "parent_male"]   = parent_male_cand
        n_parent_filled = mask_parent_holes.sum()
    else:
        n_parent_filled = 0

    # plasmid treatments: only if blank
    if can_infer_tp:
        mask_tp_holes = (
            (df["slug_infer"] == slug) &
            df["treatment_plasmid_base_codes"].apply(is_blank)
        )
        df.loc[mask_tp_holes, "treatment_plasmid_base_codes"] = tp_cand
        n_tp_filled = mask_tp_holes.sum()
    else:
        n_tp_filled = 0

    # RNA treatments: only if blank
    if can_infer_tr:
        mask_tr_holes = (
            (df["slug_infer"] == slug) &
            df["treatment_rna_base_codes"].apply(is_blank)
        )
        df.loc[mask_tr_holes, "treatment_rna_base_codes"] = tr_cand
        n_tr_filled = mask_tr_holes.sum()
    else:
        n_tr_filled = 0

    slug_debug.append(
        {
            "slug": slug,
            "n_rows": n,
            "can_infer_parents": can_infer_parents,
            "parent_female_cand": parent_female_cand,
            "parent_male_cand": parent_male_cand,
            "n_parent_filled": n_parent_filled,
            "can_infer_tp": can_infer_tp,
            "tp_cand": tp_cand,
            "n_tp_filled": n_tp_filled,
            "can_infer_tr": can_infer_tr,
            "tr_cand": tr_cand,
            "n_tr_filled": n_tr_filled,
        }
    )

slug_debug_df = pd.DataFrame(slug_debug)
slug_debug_df.head(20)

,slug,n_rows,can_infer_parents,parent_female_cand,parent_male_cand,n_parent_filled,can_infer_tp,tp_cand,n_tp_filled,can_infer_tr,tr_cand,n_tr_filled
0,Aang_Foundation/20250808_mem_organelle,21,True,membrane mChilada,membrane mChilada,0,False,None,0,True,MGCO-01,0
1,Aang_Foundation/Denoising,10,False,None,None,0,False,None,0,False,None,0
2,Korra_Foundation/20250428_mem_histone,2,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),0,False,None,0,True,pDQM117,0
3,Korra_Foundation/20250429_mem_cytosol,8,True,membrane Halo,membrane Halo,0,True,pDQM140,0,False,None,0
4,Korra_Foundation/20250513_skittles,44,False,None,None,0,False,None,0,False,None,0
5,Korra_Foundation/20250520_mem_histone,4,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),0,False,None,0,True,pDQM117,0
6,Korra_Foundation/20250521_skittles_no-membrane,5,True,Dennis (F2 of allele 318),Abe (F2 of allele 309),0,False,None,0,False,None,0
7,Korra_Foundation/20250522_skittlez,19,False,None,None,0,False,None,0,False,None,0
8,Korra_Foundation/20250523_mem_histone,5,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),0,False,None,0,True,pDQM117,0
9,Korra_Foundation/20250528_mem-histone,3,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),0,False,None,0,True,pDQM117,0


In [88]:
# cell 3b
# CELL: Hard-code skittles treatment base codes for treatment holes (v6)

import math

# canonical skittles treatment (you can adjust)
SKITTLES_TP = "pDQM034,pDQM036"

def is_blank(val: object) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    return str(val).strip() == ""

# find skittles-family rows
mask_sk = df["slug_infer"].str.contains("skittle", case=False, na=False)

# define treatment holes = both plasmid & RNA missing
mask_hole = (
    mask_sk &
    df["treatment_plasmid_base_codes"].apply(is_blank) &
    df["treatment_rna_base_codes"].apply(is_blank)
)

print("[v6 hard-code] skittles rows:", mask_sk.sum())
print("[v6 hard-code] skittles treatment holes (before):", mask_hole.sum())

# fill only holes — do not touch correctly-loaded rows
df.loc[mask_hole, "treatment_plasmid_base_codes"] = SKITTLES_TP

# quick check on known problematic ROIs
targets = [
    "/clusterfs/vast/abcabc/Korra_Foundation/20250522_skittlez/fish1/roi1",
    "/clusterfs/vast/abcabc/Korra_Foundation/20250513_skittles/fish4/roi2",
]

print("\n[v6 hard-code] Targets after fill:")
display(
    df[df["roi_dir"].isin(targets)][[
        "roi_dir",
        "slug_infer",
        "treatment_plasmid_base_codes",
        "treatment_rna_base_codes",
    ]]
)

[v6 hard-code] skittles rows: 70
[v6 hard-code] skittles treatment holes (before): 70

[v6 hard-code] Targets after fill:


,roi_dir,slug_infer,treatment_plasmid_base_codes,treatment_rna_base_codes
19,/clusterfs/vast/abcabc/Korra_Foundation/202505...,Korra_Foundation/20250522_skittlez,"pDQM034,pDQM036",NaN
922,/clusterfs/vast/abcabc/Korra_Foundation/202505...,Korra_Foundation/20250513_skittles,"pDQM034,pDQM036",NaN


In [89]:
# CELL 4: Skittles-family treatment hole filler (opinionated, holes-only)

sk = df[df["slug_infer"].str.contains("skittle", case=False, na=False)].copy()
print("[v6 skittles] skittle-family rows:", len(sk))

tp_non = sk["treatment_plasmid_base_codes"].dropna().astype(str).str.strip()
tp_non = tp_non[~tp_non.apply(is_blank)]

if tp_non.empty:
    print("[v6 skittles] No non-blank skittles treatment_plasmid_base_codes found; skipping fill.")
else:
    skittle_tp = tp_non.mode().iloc[0]
    print("[v6 skittles] Canonical skittles treatment_plasmid_base_codes:", skittle_tp)

    mask_sk = df["slug_infer"].str.contains("skittle", case=False, na=False)
    mask_hole = (
        mask_sk &
        df["treatment_plasmid_base_codes"].apply(is_blank) &
        df["treatment_rna_base_codes"].apply(is_blank)
    )

    n_holes = mask_hole.sum()
    df.loc[mask_hole, "treatment_plasmid_base_codes"] = skittle_tp
    print(f"[v6 skittles] Filled {n_holes} skittles treatment holes with", skittle_tp)

    # check the two problematic ROIs
    targets = [
        "/clusterfs/vast/abcabc/Korra_Foundation/20250522_skittlez/fish1/roi1",
        "/clusterfs/vast/abcabc/Korra_Foundation/20250513_skittles/fish4/roi2",
    ]
    print("\n[v6 skittles] Targets after fill:")
    display(
        df[df["roi_dir"].isin(targets)][[
            "roi_dir",
            "slug_infer",
            "treatment_plasmid_base_codes",
            "treatment_rna_base_codes",
        ]]
    )

[v6 skittles] skittle-family rows: 70
[v6 skittles] Canonical skittles treatment_plasmid_base_codes: pDQM034,pDQM036
[v6 skittles] Filled 0 skittles treatment holes with pDQM034,pDQM036

[v6 skittles] Targets after fill:


,roi_dir,slug_infer,treatment_plasmid_base_codes,treatment_rna_base_codes
19,/clusterfs/vast/abcabc/Korra_Foundation/202505...,Korra_Foundation/20250522_skittlez,"pDQM034,pDQM036",NaN
922,/clusterfs/vast/abcabc/Korra_Foundation/202505...,Korra_Foundation/20250513_skittles,"pDQM034,pDQM036",NaN


In [90]:
# CELL 5: Load constructs and tags (autoload kit) for v6

AUTOLOAD = BASE / "seed_kits" / "2025-11-15-121231-autoload"

constructs_path = AUTOLOAD / "constructs_plasmid.csv"
tags_path       = AUTOLOAD / "tags.xlsx"

df_constructs = pd.read_csv(constructs_path)
df_tags       = pd.read_excel(tags_path)

cons_small = (
    df_constructs[["plasmid_code", "fluor_code", "tag_code", "tag_pos"]]
    .rename(columns={"plasmid_code": "plasmid_base_code"})
    .copy()
)
cons_small["plasmid_base_code"] = cons_small["plasmid_base_code"].astype(str).str.strip()

if "tag_code" in df_tags.columns:
    tag_code_col = "tag_code"
elif "nickname" in df_tags.columns:
    tag_code_col = "nickname"
else:
    raise RuntimeError("No tag_code or nickname column in tags.xlsx")

loc_cols = [c for c in df_tags.columns if c.lower() == "localization"]
if not loc_cols:
    raise RuntimeError("No localization column in tags.xlsx")
loc_col = loc_cols[0]

tags_map = df_tags[[tag_code_col, loc_col]].copy()
tags_map.columns = ["tag_code", "localization"]
tags_map["tag_code"] = tags_map["tag_code"].astype(str).str.strip()

print("cons_small:", cons_small.shape, "tags_map:", tags_map.shape)

cons_small: (286, 4) tags_map: (10, 2)


In [91]:
# CELL 6: Helpers for fusions + localizations

def split_codes(val: object) -> list[str]:
    if val is None or pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [c.strip() for c in re.split(r"[;,]", s) if c.strip()]

def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if x is None or pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(" ".join(s.split()) for s in vals))
    return ",".join(uniq)

def make_fusion_label(row: pd.Series) -> str | None:
    def norm(val: object) -> str:
        if val is None or pd.isna(val):
            return ""
        return str(val).strip()
    fluor = norm(row.get("fluor_code"))
    tag   = norm(row.get("tag_code"))
    pos   = norm(row.get("tag_pos"))
    if not fluor and not tag:
        return None
    if fluor and not tag:
        return fluor
    if not fluor and tag:
        return tag
    if fluor and tag and pos:
        return f"{fluor}::{tag}({pos})"
    return f"{fluor}::{tag}"

def fluor_loc_label(row: pd.Series) -> str | None:
    def norm(val: object) -> str:
        if val is None or pd.isna(val):
            return ""
        return str(val).strip()
    fluor = norm(row.get("fluor_code"))
    loc   = norm(row.get("localization"))
    if not fluor:
        return None
    if loc:
        return f"{fluor}({loc})"
    return fluor

def fusion_loc_label(row: pd.Series) -> str | None:
    fusion = make_fusion_label(row)
    if not fusion:
        return None
    loc = row.get("localization")
    if loc is None or pd.isna(loc) or str(loc).strip() == "":
        return fusion
    return f"{fusion}({str(loc).strip()})"

In [92]:
# CELL 7: Genotype fusions + localizations + fluor_loc/fusion_loc (v6)

gexp = (
    df[["roi_dir", "genotype_base_codes"]]
    .dropna(subset=["genotype_base_codes"])
    .assign(code_list=lambda d: d["genotype_base_codes"].apply(split_codes))
    .explode("code_list")
    .rename(columns={"code_list": "plasmid_base_code"})
)
gexp["plasmid_base_code"] = gexp["plasmid_base_code"].astype(str).str.strip()

gjoin = gexp.merge(cons_small, how="left", on="plasmid_base_code")
gjoin_loc = gjoin.merge(tags_map, how="left", on="tag_code")

gjoin_loc["fusion_label"] = gjoin_loc.apply(make_fusion_label, axis=1)
gjoin_loc["fluor_loc"]    = gjoin_loc.apply(fluor_loc_label, axis=1)
gjoin_loc["fusion_loc"]   = gjoin_loc.apply(fusion_loc_label, axis=1)

g_per_roi = (
    gjoin_loc.groupby("roi_dir", as_index=False)
    .agg(
        genotype_marker_fusion_labels     = ("fusion_label", agg_uniq),
        genotype_marker_localizations     = ("localization", agg_uniq),
        genotype_marker_fluor_loc_labels  = ("fluor_loc", agg_uniq),
        genotype_marker_fusion_loc_labels = ("fusion_loc", agg_uniq),
    )
)

df = df.merge(g_per_roi, how="left", on="roi_dir")

preview_cols = ["roi_dir"]
for c in [
    "genotype_marker_fusion_labels",
    "genotype_marker_localizations",
    "genotype_marker_fluor_loc_labels",
    "genotype_marker_fusion_loc_labels",
]:
    if c in df.columns:
        preview_cols.append(c)

df[preview_cols].head(10)

,roi_dir,genotype_marker_fusion_labels,genotype_marker_localizations,genotype_marker_fluor_loc_labels,genotype_marker_fusion_loc_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG::2xLynk(N),membrane,tdmSG(membrane),tdmSG::2xLynk(N)(membrane)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG::2xLynk(N),membrane,tdmSG(membrane),tdmSG::2xLynk(N)(membrane)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,None,None


In [94]:
# CELL7b: Infer anatomy from imaging sheet and merge into v6 (not optional)

import re

# path to imaging sheet
sheet_path = BASE / "seed_kits" / "legacy_wrangling" / "raw" / "2025-11-13-124226-imaging_sheet.xlsx"
df_sheet = pd.read_excel(sheet_path)

print("df_sheet:", df_sheet.shape)

# normalize Data location and compute sheet_slug similar to slug_infer
df_sheet["Data location"] = df_sheet["Data location"].astype(str).str.strip()

def sheet_slug_from_data_location(path: str) -> str:
    p = path.replace("\\", "/").rstrip("/")
    parts = p.split("/")
    # e.g. X:/abcabc/Korra_Foundation/20251110_mem-histone
    if len(parts) >= 3:
        foundation = parts[-2]  # Korra_Foundation
        experiment = parts[-1]  # 20251110_mem-histone
        return f"{foundation}/{experiment}"
    return p

df_sheet["sheet_slug"] = df_sheet["Data location"].apply(sheet_slug_from_data_location)

# clean text columns
def clean_list_text(val: object) -> str | None:
    if val is None or pd.isna(val):
        return None
    s = str(val)
    s = s.replace("\u00a0", " ")  # non-breaking space
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

for col in ["Imaged Locations", "Unique Targets"]:
    if col in df_sheet.columns:
        df_sheet[col] = df_sheet[col].apply(clean_list_text)

def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if x is None or pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(vals))
    return "; ".join(uniq)

anatomy_agg = (
    df_sheet.groupby("sheet_slug", as_index=False)
            .agg(
                anatomy_imaged_locations=("Imaged Locations", agg_uniq),
                anatomy_targets=("Unique Targets", agg_uniq),
            )
)

print("anatomy_agg:", anatomy_agg.shape)

# merge anatomy into df via slug_infer ~ sheet_slug
df = df.merge(
    anatomy_agg,
    how="left",
    left_on="slug_infer",
    right_on="sheet_slug"
)

df = df.drop(columns=["sheet_slug"], errors="ignore")

# quick sanity check
df.loc[
    df["slug_infer"].str.contains("mem_histone|skittle", case=False, na=False),
    ["roi_dir", "slug_infer", "anatomy_imaged_locations", "anatomy_targets"]
].head(20)

df_sheet: (306, 20)
anatomy_agg: (220, 3)


KeyError: "['anatomy_imaged_locations', 'anatomy_targets'] not in index"

In [95]:
# CELL 8: Treatment fusions + localizations + fluor_loc/fusion_loc (v6)

def collect_all_treatment_codes(row: pd.Series) -> list[str]:
    codes: list[str] = []
    codes.extend(split_codes(row.get("treatment_plasmid_base_codes")))
    codes.extend(split_codes(row.get("treatment_rna_base_codes")))
    seen = set()
    out: list[str] = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

texp = (
    df[["roi_dir", "treatment_plasmid_base_codes", "treatment_rna_base_codes"]]
    .assign(code_list=lambda d: d.apply(collect_all_treatment_codes, axis=1))
    .explode("code_list")
)

texp = texp[texp["code_list"].notna() & (texp["code_list"].astype(str).str.strip() != "")]
texp = texp.rename(columns={"code_list": "plasmid_base_code"})
texp["plasmid_base_code"] = texp["plasmid_base_code"].astype(str).str.strip()

tjoin = texp.merge(cons_small, how="left", on="plasmid_base_code")
tjoin_loc = tjoin.merge(tags_map, how="left", on="tag_code")

tjoin_loc["fusion_label"] = tjoin_loc.apply(make_fusion_label, axis=1)
tjoin_loc["fluor_loc"]    = tjoin_loc.apply(fluor_loc_label, axis=1)
tjoin_loc["fusion_loc"]   = tjoin_loc.apply(fusion_loc_label, axis=1)

t_per_roi = (
    tjoin_loc.groupby("roi_dir", as_index=False)
    .agg(
        treatment_marker_fusion_labels     = ("fusion_label", agg_uniq),
        treatment_marker_localizations     = ("localization", agg_uniq),
        treatment_marker_fluor_loc_labels  = ("fluor_loc", agg_uniq),
        treatment_marker_fusion_loc_labels = ("fusion_loc", agg_uniq),
    )
)

df = df.merge(t_per_roi, how="left", on="roi_dir")

preview_cols = ["roi_dir"]
for c in [
    "treatment_plasmid_base_codes",
    "treatment_marker_fusion_labels",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
    "treatment_marker_fusion_loc_labels",
]:
    if c in df.columns:
        preview_cols.append(c)

df[preview_cols].head(10)

,roi_dir,treatment_plasmid_base_codes,treatment_marker_fusion_labels,treatment_marker_localizations,treatment_marker_fluor_loc_labels,treatment_marker_fusion_loc_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,mScarlet3S2::H2B(N),histone,mScarlet3S2(histone),mScarlet3S2::H2B(N)(histone)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,mScarlet3S2::H2B(N),histone,mScarlet3S2(histone),mScarlet3S2::H2B(N)(histone)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,mGold2s,None,mGold2s,mGold2s


In [81]:
# CELL 9: Build all_unique_organelles from tag localizations + folder path (no sheet anatomy)

import re
import pandas as pd
import math

def split_locs(val: object) -> list[str]:
    """Split comma/semicolon separated localization strings into clean tokens."""
    if val is None or pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    s = s.replace(";", ",")
    return [p.strip() for p in s.split(",") if p.strip()]

def infer_organelle_from_path(path: str) -> list[str]:
    """Infer organelle-ish labels from roi_dir / slug naming."""
    p = str(path).lower()
    organelles: list[str] = []

    # heuristic patterns — adjust as you like
    if "mem_" in p or "mem-" in p or "mem " in p:
        organelles.append("membrane")
    if "mito" in p:
        organelles.append("mitochondria")
    if "histone" in p:
        organelles.append("histone")
    if "peroxi" in p:
        organelles.append("peroxisome")
    if re.search(r"\ber[_\-]", p):  # er_ or er- after word boundary
        organelles.append("ER")

    # drop any junk like 'skittles test' implicit from path
    # we only return the organelle tokens above
    return organelles

def combine_unique_lists(*args) -> str | None:
    uniq = []
    seen = set()
    for lst in args:
        for item in lst:
            if not item:
                continue
            if item not in seen:
                seen.add(item)
                uniq.append(item)
    return ",".join(sorted(uniq)) if uniq else None

def row_all_organelles(r: pd.Series) -> str | None:
    pieces: list[list[str]] = []

    # from tag localizations
    if "genotype_marker_localizations" in r.index:
        pieces.append(split_locs(r.get("genotype_marker_localizations")))
    if "treatment_marker_localizations" in r.index:
        pieces.append(split_locs(r.get("treatment_marker_localizations")))

    # from file path
    pieces.append(infer_organelle_from_path(r.get("roi_dir")))

    return combine_unique_lists(*pieces)

df["all_unique_organelles"] = df.apply(row_all_organelles, axis=1)

# quick sanity check: show some mem_histone + skittle rows
df.loc[
    df["roi_dir"].str.contains("mem_histone|skittle", case=False, na=False),
    ["roi_dir", "genotype_marker_localizations", "treatment_marker_localizations", "all_unique_organelles"]
].head(30)

,roi_dir,genotype_marker_localizations,treatment_marker_localizations,anatomy_imaged_locations,anatomy_targets,all_unique_organelles
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane,histone,NaN,NaN,"histone,membrane"
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane,histone,NaN,NaN,"histone,membrane"
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,None,NaN,NaN,None


In [82]:
# CELL 10: Normalize genotype_pretty to ASCII (no ×, no weird symbols)

def clean_genotype_pretty(val: object) -> str | None:
    if val is None or pd.isna(val):
        return None
    s = str(val)
    s = s.replace("×", " x ")
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

df["genotype_pretty"] = df["genotype_pretty"].apply(clean_genotype_pretty)
df[["genotype_pretty"]].head(10)

,genotype_pretty
0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...
1,ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...
2,membrane Halo x membrane Halo
3,membrane Halo x membrane Halo
4,membrane Halo x membrane Halo
5,membrane Halo x membrane Halo
6,membrane Halo x membrane Halo
7,membrane Halo x membrane Halo
8,membrane Halo x membrane Halo
9,membrane Halo x membrane Halo


In [83]:
# CELL 11: Write imaging_roi_annotations_AUTO_v6.csv with full v6 schema

annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
    "genotype_marker_localizations",
    "genotype_marker_fluor_loc_labels",
    "genotype_marker_fusion_loc_labels",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
    "treatment_marker_fusion_loc_labels",
    "all_marker_fluor_codes",
    "all_unique_organelles",
    "plate_id_filled", "slot_id_filled",
    "roi_index_within_slot", "roi_code",
]

annot_cols = [c for c in annot_cols if c in df.columns]

annot_v6 = df[annot_cols].copy()

annot_path_v6 = out_csv("imaging_roi_annotations_AUTO")
annot_v6.to_csv(annot_path_v6, index=False)

annot_path_v6, annot_v6.shape

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO_v6.csv'),
 (976, 26))